# 🛡️ AEGIS — analise a segurança do seu código

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/felpsbc/SecTool/blob/main/AEGIS.ipynb)

### Como usar (3 passos)

1. Clique no botão **▶** da célula abaixo e espere ~1 minuto (só na primeira vez).
2. **Cole o seu código** na caixa que aparecer e escolha a linguagem.
3. Clique em **Analisar**.

O AEGIS aponta vulnerabilidades do OWASP Top 10 (SQL Injection, XSS, senhas no código…) e explica **como corrigir** cada uma.
O seu código **nunca é executado** — ele só é lido — e fica numa máquina temporária do Google, apagada quando a sessão termina.

> Se aparecer o aviso *"Este notebook não foi criado pelo Google"*, clique em **Executar assim mesmo**.

In [ ]:
#@title ▶ **Clique aqui para abrir o AEGIS** (cole seu código na caixa que vai aparecer)
import html, os, shutil, subprocess, sys, tempfile
from pathlib import Path
from IPython.display import HTML, clear_output, display

REPO_AEGIS = "https://github.com/felpsbc/SecTool.git"

def _token():
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

def _com_token(url):
    token = _token()
    if token and url.startswith("https://github.com/"):
        return url.replace("https://", f"https://x-access-token:{token}@", 1)
    return url

def _sem_token(texto):
    token = _token()
    return texto.replace(token, "***") if token else texto

# --- Instalacao -------------------------------------------------------------
print("Preparando o AEGIS... (cerca de 1 minuto na primeira vez)")
# Rodando de um clone local (Jupyter/VS Code)? Usa o proprio repositorio.
_aqui = Path.cwd()
_locais = [p for p in (_aqui, *_aqui.parents) if (p / "TOOL" / "owaspscan").is_dir()]
if _locais:
    RAIZ_AEGIS = _locais[0]
else:
    RAIZ_AEGIS = Path("/content/AEGIS") if Path("/content").is_dir() else _aqui / "AEGIS"
    if not RAIZ_AEGIS.exists():
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_AEGIS, str(RAIZ_AEGIS)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("Nao consegui baixar o AEGIS. Confira a internet e rode de novo.\n" + r.stderr)
TOOL = RAIZ_AEGIS / "TOOL"
# Extra [linguagens] = tree-sitter para JS/TS, Java, PHP e HTML.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{TOOL}[linguagens]", "ipywidgets"], check=True)
if str(TOOL) not in sys.path:
    sys.path.insert(0, str(TOOL))

from owaspscan.varredura import VERSAO, executar
from owaspscan.relatorio import ordenar, para_dict

# --- Exibicao do resultado ------------------------------------------------
ULTIMO_RELATORIO = None
_CORES = {"alta": "#d93025", "media": "#e37400", "baixa": "#5f6368"}

def mostrar_resultado(varredura):
    """Mostra o resultado como cartoes: problema, trecho e como corrigir."""
    global ULTIMO_RELATORIO
    ULTIMO_RELATORIO = para_dict(varredura)
    e = html.escape
    achados = ordenar(varredura.achados)
    contagem = {s: sum(a.severidade == s for a in achados) for s in ("alta", "media", "baixa")}

    if varredura.tem_severidade_alta:
        veredito, cor = "⚠️ Encontramos falhas de severidade ALTA — corrija antes de publicar.", _CORES["alta"]
    elif achados:
        veredito, cor = "🔎 Nenhuma falha alta, mas há pontos para revisar.", _CORES["media"]
    elif varredura.falhas and not varredura.arquivos_analisados:
        veredito, cor = "❌ Não foi possível ler o código (veja o motivo abaixo).", _CORES["baixa"]
    else:
        veredito, cor = "✅ Nenhuma vulnerabilidade encontrada pelos detectores do AEGIS.", "#188038"

    partes = [f"""
<div style="font-family:system-ui,sans-serif;line-height:1.45;max-width:980px">
<div style="border-left:6px solid {cor};padding:10px 14px;margin:8px 0;border-radius:4px;background:rgba(127,127,127,.08)">
  <div style="font-size:17px;font-weight:600">{veredito}</div>
  <div style="margin-top:4px">{len(achados)} achado(s): <b style="color:{_CORES['alta']}">{contagem['alta']} alta</b> ·
  <b style="color:{_CORES['media']}">{contagem['media']} média</b> · <b>{contagem['baixa']} baixa</b>
  — {varredura.arquivos_analisados + varredura.arquivos_de_configuracao} arquivo(s) analisado(s)</div>
</div>"""]

    for f in varredura.falhas:
        linha = f", linha {f.linha}" if f.linha else ""
        partes.append(f'<div style="color:{_CORES["baixa"]};margin:4px 0">Não foi possível ler <code>{e(f.arquivo)}</code>'
                      f' ({e(f.motivo)}{linha}): {e(f.mensagem)}</div>')

    for i, a in enumerate(achados, 1):
        c = _CORES.get(a.severidade, _CORES["baixa"])
        aviso = ('<div style="font-size:13px;opacity:.8;margin-top:4px">Candidato: padrão suspeito que '
                 '<b>precisa de revisão manual</b> — não é certeza.</div>') if a.confianca == "baixa" else ""
        partes.append(f"""
<div style="border:1px solid rgba(127,127,127,.35);border-left:6px solid {c};border-radius:6px;padding:10px 14px;margin:10px 0">
  <div><span style="background:{c};color:#fff;border-radius:4px;padding:1px 8px;font-size:12px;font-weight:600">
  {e(a.severidade.upper())}</span> <b>{i}. {e(a.vulnerabilidade)}</b> ({e(a.cwe)}) — confiança {e(a.confianca)}</div>
  <div style="font-size:13px;opacity:.8;margin-top:2px">{e(a.arquivo)}, linha {a.linha} · {e(a.owasp)}</div>
  <pre style="background:rgba(127,127,127,.12);padding:6px 10px;border-radius:4px;white-space:pre-wrap;margin:8px 0">{e(a.trecho)}</pre>
  <div><b>Problema:</b> {e(a.descricao)}</div>
  <div style="margin-top:4px"><b>Como corrigir:</b> {e(a.recomendacao)}</div>{aviso}
</div>""")

    if not achados:
        partes.append('<div style="opacity:.8">O AEGIS procura uma lista definida de falhas (OWASP Top 10). '
                      '"Nenhum achado" não garante que o código é seguro.</div>')
    partes.append("</div>")
    display(HTML("".join(partes)))

# --- Caixa para colar o codigo ----------------------------------------------
_ARQUIVO_POR_LINGUAGEM = {
    "Python": "codigo.py", "JavaScript": "codigo.js", "TypeScript": "codigo.ts",
    "Java": "Codigo.java", "PHP": "codigo.php", "HTML": "pagina.html",
    "Configuração .env": ".env", "Configuração YAML": "config.yml",
}

def analisar_codigo(codigo, linguagem="Python"):
    """Analisa um trecho de codigo colado e mostra o resultado."""
    pasta = Path(tempfile.mkdtemp(prefix="aegis_codigo_"))
    try:
        arquivo = pasta / _ARQUIVO_POR_LINGUAGEM[linguagem]
        arquivo.write_text(codigo, encoding="utf-8")
        mostrar_resultado(executar(arquivo))
    finally:
        shutil.rmtree(pasta, ignore_errors=True)

clear_output()
try:
    import ipywidgets as w

    linguagem = w.Dropdown(options=list(_ARQUIVO_POR_LINGUAGEM), value="Python", description="Linguagem:")
    caixa = w.Textarea(placeholder="Cole aqui o seu código...",
                       layout=w.Layout(width="100%", height="320px"))
    botao = w.Button(description="Analisar", button_style="primary", icon="shield")
    exemplo = w.Button(description="Usar um exemplo vulnerável", layout=w.Layout(width="auto"))
    limpar = w.Button(description="Limpar")
    resultado = w.Output()

    def _analisar(_):
        resultado.clear_output()
        with resultado:
            if not caixa.value.strip():
                print("Cole um código na caixa acima antes de clicar em Analisar.")
                return
            analisar_codigo(caixa.value, linguagem.value)

    def _exemplo(_):
        linguagem.value = "Python"
        caixa.value = (RAIZ_AEGIS / "docs" / "teste" / "app_vulneravel.py").read_text(encoding="utf-8")

    def _limpar(_):
        caixa.value = ""
        resultado.clear_output()

    botao.on_click(_analisar)
    exemplo.on_click(_exemplo)
    limpar.on_click(_limpar)
    display(HTML(f"<b>AEGIS {VERSAO} pronto.</b> Cole o código, escolha a linguagem e clique em <b>Analisar</b>."),
            linguagem, caixa, w.HBox([botao, exemplo, limpar]), resultado)
except ImportError:
    print(f"AEGIS {VERSAO} pronto. Sem ipywidgets, use: analisar_codigo(codigo, 'Python')")

---
## Analisar um projeto inteiro (opcional)

Para analisar **todos os arquivos** de um projeto de uma vez, escolha a origem no formulário à direita e clique em **▶**:

| Origem | O que preencher |
| --- | --- |
| `github` | `LINK_GITHUB` = link do repositório. Privado? Salve um token do GitHub em 🔑 **Secrets** (barra lateral) com o nome `GITHUB_TOKEN`. |
| `zip` | Nada — aparece um botão para enviar o `.zip` do projeto. |
| `google_drive` | `PASTA` = caminho dentro do *Meu Drive*, ex.: `projetos/minha-api`. O Colab pede permissão para acessar o Drive. |
| `pasta_local` | `PASTA` = caminho completo nesta máquina (uso em Jupyter local). |

In [ ]:
#@title ▶ Analisar um projeto inteiro
ORIGEM = "github"  #@param ["github", "zip", "google_drive", "pasta_local"]
LINK_GITHUB = "https://github.com/usuario/projeto"  #@param {type:"string"}
PASTA = ""  #@param {type:"string"}

import zipfile

TRABALHO = Path(tempfile.gettempdir()) / "aegis_projeto"
shutil.rmtree(TRABALHO, ignore_errors=True)
TRABALHO.mkdir(parents=True)

def _extrair_zip(arquivo, destino):
    with zipfile.ZipFile(arquivo) as z:
        for membro in z.namelist():  # evita zip slip ("../" saindo do destino)
            if not (destino / membro).resolve().is_relative_to(destino.resolve()):
                raise ValueError(f"zip com caminho suspeito: {membro}")
        z.extractall(destino)

if ORIGEM == "github":
    if "usuario/projeto" in LINK_GITHUB:
        raise ValueError("Troque LINK_GITHUB pelo link do seu repositório no formulário à direita.")
    PROJETO = TRABALHO / "repo"
    print("Baixando o repositório...")
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", _com_token(LINK_GITHUB.strip()), str(PROJETO)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("Não consegui baixar o repositório. Confira o link (e o GITHUB_TOKEN, se for privado).\n"
                           + _sem_token(r.stderr))
elif ORIGEM == "zip":
    from google.colab import files
    PROJETO = TRABALHO / "zip"
    print("Escolha o arquivo .zip do seu projeto:")
    for nome, dados in files.upload().items():
        (TRABALHO / nome).write_bytes(dados)
        _extrair_zip(TRABALHO / nome, PROJETO)
elif ORIGEM == "google_drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJETO = Path("/content/drive/MyDrive") / PASTA.strip().strip("/")
else:
    PROJETO = Path(PASTA).expanduser()

if not PROJETO.exists():
    raise FileNotFoundError(f"Caminho não encontrado: {PROJETO}")

clear_output()
print(f"Projeto analisado: {PROJETO}")
mostrar_resultado(executar(PROJETO))

In [ ]:
#@title ▶ Baixar o último relatório (JSON)
import json
if ULTIMO_RELATORIO is None:
    print("Analise um código ou projeto primeiro.")
else:
    saida = Path(tempfile.gettempdir()) / "relatorio_aegis.json"
    saida.write_text(json.dumps(ULTIMO_RELATORIO, ensure_ascii=False, indent=2), encoding="utf-8")
    try:
        from google.colab import files
        files.download(str(saida))
    except ImportError:
        print(f"Relatório salvo em {saida}")

---
## Métricas: a ferramenta funciona? (opcional)

O repositório traz um **corpus rotulado**: cada vulnerabilidade plantada tem um comentário `# achado: CWE-xx`
na linha exata. As células abaixo rodam o AEGIS nesse corpus e comparam com o gabarito.

- **Precisão** = dos achados apontados, quantos são reais (falso positivo derruba).
- **Recall** = das vulnerabilidades reais, quantas foram encontradas (falso negativo derruba).
- `FN conh.` = limitações conhecidas e declaradas da ferramenta.

In [ ]:
#@title ▶ Métricas no corpus rotulado
from owaspscan.metricas import avaliar, formatar_tabela

corpus = TOOL / "tests" / "amostras"
v = executar(corpus)
print(formatar_tabela(avaliar(corpus, v), v))

In [ ]:
#@title ▶ Prova rápida: código vulnerável vs. código corrigido
amostras = TOOL / "tests" / "amostras"
for nome in ("vulneravel", "seguro"):
    v = executar(amostras / nome)
    print(f"{nome:>10}: {len(v.achados):3d} achados em {v.arquivos_analisados} arquivos")
print("\nEsperado: bem menos achados em 'seguro'. Os que restam estão previstos no gabarito")
print("(comentários '# achado:' nos próprios arquivos) e quase todos são candidatos de confiança baixa.")

---
Encontrou um falso positivo ou uma vulnerabilidade que o AEGIS deixou passar? Abra uma issue em
<https://github.com/felpsbc/SecTool/issues> com o trecho de código — isso alimenta o corpus de avaliação.